# Sesi 11 — RAG bagian 3: Merangkai Semuanya


**Alur proses RAG kita**

```
ingest  ->  embedding  ->  indexing  ->  retrieval  ->  re-ranking
        ->  context injection  ->  generate
```

In [ ]:
import os
import pathlib
import re

import faiss
import numpy as np
from dotenv import load_dotenv
from google import genai
from google.genai import types
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

load_dotenv()

MODEL_EMBED = "gemini-embedding-001"
MODEL_JAWAB = "gemini-2.5-flash"
MODEL_RERANK = "cross-encoder/mmarco-mMiniLMv2-L12-H384-v1"
DIMENSI = 768
UKURAN_CHUNK = 600


def akar_sesi():
    # Folder yang memuat kb_lindungiku, entah folder ini atau folder induknya.
    for p in (pathlib.Path("."), pathlib.Path("..")):
        if (p / "kb_lindungiku").is_dir():
            return p.resolve()
    raise SystemExit(
        "Folder kb_lindungiku tidak ditemukan. Jalankan notebook ini dari folder "
        "Session 11 atau dari salah satu subfoldernya."
    )


FOLDER_KB = akar_sesi() / "kb_lindungiku"

client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])
reranker = CrossEncoder(MODEL_RERANK)
print("knowledge base :", FOLDER_KB)
print("siap")

## Bagian 1 — Membangun index

In [ ]:
def bersihkan(teks):
    teks = teks.replace("\u00a0", " ")
    teks = re.sub(r"[ \t]+", " ", teks)
    return re.sub(r"\n{3,}", "\n\n", teks).strip()


def normalisasi(v):
    panjang = np.linalg.norm(v, axis=1, keepdims=True)
    panjang[panjang == 0] = 1e-12
    return v / panjang


def embed(daftar_teks, task_type, batas=100):
    hasil = []
    for i in range(0, len(daftar_teks), batas):
        resp = client.models.embed_content(
            model=MODEL_EMBED,
            contents=daftar_teks[i : i + batas],
            config=types.EmbedContentConfig(task_type=task_type, output_dimensionality=DIMENSI),
        )
        hasil.extend(e.values for e in resp.embeddings)
    return np.array(hasil, dtype="float32")


pemisah = RecursiveCharacterTextSplitter(
    chunk_size=UKURAN_CHUNK,
    chunk_overlap=80,
    separators=["\n## ", "\n\n", "\n", ". ", " ", ""],
)

potongan = []
for berkas in sorted(FOLDER_KB.glob("*.md")):
    for teks in pemisah.split_text(bersihkan(berkas.read_text(encoding="utf-8"))):
        potongan.append({"sumber": berkas.name, "teks": teks})

index = faiss.IndexFlatIP(DIMENSI)
index.add(normalisasi(embed([p["teks"] for p in potongan], "RETRIEVAL_DOCUMENT")))

print(f"{len(potongan)} chunk, {index.ntotal} vector. index siap.")

## Bagian 2 — Context injection

Di sinilah retrieval berubah menjadi jawaban, dan di sinilah sebagian besar sistem
RAG gagal secara diam-diam.

Kalau Anda hanya menempelkan potongan ke dalam prompt tanpa instruksi, model akan
menganggapnya sebagai bahan tambahan, bukan sebagai satu-satunya sumber. Ketika
konteksnya tidak memuat jawaban, model kembali memakai data latihnya dan
mengarang, dengan nada yang sama meyakinkannya.

Prompt di bawah melakukan tiga hal, dan ketiganya diperlukan.

1. **Membatasi sumber.** Kata HANYA ditulis dengan huruf besar karena memang
   sepenting itu.
2. **Memberi jalan keluar.** Model diberi izin eksplisit untuk mengaku tidak tahu.
   Tanpa kalimat ini, model merasa wajib menjawab apa pun.
3. **Memisahkan bagian dengan jelas.** Konteks dan pertanyaan dipisahkan oleh
   penanda, supaya model tidak salah menebak mana yang mana.

Ini adalah structured prompt yang sama seperti yang Anda pelajari sebelumnya,
dengan konteks hasil retrieval di dalamnya.

In [ ]:
# TODO 1: tulis template prompt-nya.
# Harus memuat tiga hal: batasi model hanya pada konteks, beri izin untuk
# mengaku tidak tahu, dan pisahkan KONTEKS dari PERTANYAAN dengan penanda jelas.
# Sediakan dua placeholder, yaitu {konteks} dan {pertanyaan}.
TEMPLATE = """"""


def susun_prompt(pertanyaan, potongan_terpilih):
    # TODO 2: gabungkan potongan terpilih menjadi satu string konteks.
    # Sertakan nama sumber tiap potongan, dan pisahkan antar potongan dengan
    # pembatas yang jelas. Lalu isikan ke dalam TEMPLATE.
    pass


print(susun_prompt("contoh pertanyaan", potongan[:2]))

## Bagian 3 — Fungsi `jawab`

Seluruh pipeline dalam satu fungsi. Baca dari atas ke bawah, dan Anda akan melihat
ketujuh langkahnya berurutan.

Perhatikan pola dua tahapnya. `k_ambil=8` mengambil delapan kandidat dengan
pencarian yang murah, lalu `k_pakai=3` menyisakan tiga terbaik setelah diurutkan
ulang oleh cross-encoder. Mengambil lebih banyak dari yang dipakai adalah
intinya, karena re-ranker hanya bisa memperbaiki urutan dari kandidat yang
diberikan kepadanya.

In [ ]:
def jawab(pertanyaan, k_ambil=8, k_pakai=3, tampilkan=True):
    # 1. pertanyaan menjadi vector
    q = normalisasi(embed([pertanyaan], "RETRIEVAL_QUERY"))

    # TODO 3: cari k_ambil kandidat terdekat di dalam index,
    # lalu ambil potongan yang sesuai dengan nomor baris hasil pencarian.
    kandidat = None

    # TODO 4: nilai setiap pasangan (pertanyaan, teks potongan) dengan reranker,
    # urutkan dari nilai tertinggi, lalu sisakan k_pakai teratas.
    terpilih = None

    prompt = susun_prompt(pertanyaan, terpilih)

    # TODO 5: kirim prompt ke model memakai client.models.generate_content
    # dengan model=MODEL_JAWAB, lalu simpan hasilnya ke resp.
    resp = None

    if tampilkan:
        print("SUMBER YANG DIPAKAI")
        for p in terpilih:
            print("   -", p["sumber"])
        print()
        print("JAWABAN")
        print(resp.text.strip())
    return resp.text.strip(), terpilih


_ = jawab("Berapa lama masa tunggu klaim rawat inap?")

## Bagian 4 — Uji kejujuran

Sistem yang menjawab benar ketika jawabannya ada di dokumen itu bagus. Sistem yang
mengaku tidak tahu ketika jawabannya memang tidak ada di dokumen itu **jauh lebih
bagus**, dan jauh lebih jarang.

Pertanyaan di bawah tidak dibahas sama sekali oleh ketujuh dokumen kita.

Kalau model tetap menjawab dengan angka, berarti prompt Anda belum cukup kuat.
Perkuat kalimat izin mengaku tidak tahu, lalu jalankan ulang. Kegagalan di sini
adalah pelajaran yang lebih berguna daripada keberhasilan yang mulus.

In [ ]:
_ = jawab("Berapa premi asuransi perjalanan ke luar negeri di LindungiKu?")

## Bagian 5 — Giliran Anda

Ajukan tiga pertanyaan Anda sendiri kepada sistem yang baru selesai dibangun.

1. Satu pertanyaan yang jawabannya jelas ada di dalam dokumen
2. Satu pertanyaan yang hanya dijawab separuh oleh dokumen
3. Satu pertanyaan yang sama sekali tidak dibahas dokumen

Dua pertanyaan pertama menunjukkan sistemnya bekerja. Pertanyaan ketiga
menunjukkan sistemnya jujur.

Ketika hasilnya terlihat aneh, jangan langsung menyalahkan modelnya. Periksa dulu
potongan mana yang terambil, memakai sel diagnosa di bawah. Hampir selalu
masalahnya ada di retrieval, bukan di generate.

In [ ]:
PERTANYAAN_SAYA = [
    "",   # 1. jelas dijawab dokumen
    "",   # 2. dijawab sebagian
    "",   # 3. tidak dibahas sama sekali
]

for p in PERTANYAAN_SAYA:
    if not p.strip():
        continue
    print("=" * 72)
    print(p)
    print("=" * 72)
    jawab(p)
    print()

### Sel diagnosa

Ketika jawabannya salah, jalankan ini dengan pertanyaan yang sama. Ia menunjukkan
apa yang sebenarnya dibaca model, sebelum dan sesudah re-ranking.

In [ ]:
def diagnosa(pertanyaan, k_ambil=8):
    q = normalisasi(embed([pertanyaan], "RETRIEVAL_QUERY"))
    skor, baris = index.search(q, k_ambil)
    kandidat = [potongan[i] for i in baris[0]]
    nilai = reranker.predict([(pertanyaan, p["teks"]) for p in kandidat])

    print(f"{'#':<3} {'cosine':>7} {'rerank':>8}  sumber")
    print("-" * 60)
    for n, (s, r, p) in enumerate(zip(skor[0], nilai, kandidat), 1):
        print(f"{n:<3} {s:>7.3f} {r:>8.2f}  {p['sumber']}")
    print()
    print("Urutan berubah setelah re-ranking? Itu tandanya cross-encoder bekerja.")


diagnosa("Berapa lama masa tunggu klaim rawat inap?")